# S-curve builder

This notebook contains an S-curve builder with visual output. The intent is to have a place to experiment with building S-curves for other parts of the project.

In [1]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def build_s_curve(start_date, end_date, resolution="1h", L=1.0, k=1.2, x0=2027, value_2024=0.527, value_2030=0.721):
    """
    Builds an S-curve over a specified time range and resolution.
    
    Parameters:
        start_date (str): Start of the time range (e.g., "2020-01-01").
        end_date (str): End of the time range (e.g., "2045-12-31").
        resolution (str): Time resolution (e.g., "1h", "1d", "1w", "1y").
        L (float): Maximum value of the curve (asymptote).
        k (float): Growth rate (controls steepness).
        x0 (float): Midpoint of the curve (year where growth is fastest).
        value_2024 (float): Value of the curve in 2024.
        value_2030 (float): Value of the curve in 2030.
    
    Returns:
        pd.Series: A time-indexed series of S-curve values.
    """
    # Generate timestamps based on resolution
    timestamps = pd.date_range(start=start_date, end=end_date, freq=resolution)
    
    # Convert timestamps to years as a float
    years = timestamps.year + (timestamps.dayofyear - 1) / 365.0
    
    # Apply logistic function to create S-curve
    s_curve = L / (1 + np.exp(-k * (years - x0)))
    
    # Scale and shift the curve to match 2024 and 2030 values
    scale = (value_2030 - value_2024) / (s_curve[np.isclose(years, 2030)].mean() - s_curve[np.isclose(years, 2024)].mean())
    shift = value_2024 - s_curve[np.isclose(years, 2024)].mean() * scale
    s_curve = s_curve * scale + shift
    
    return pd.Series(s_curve, index=timestamps)

In [3]:
# Parameters for the S-curve
start_date = "2020-01-01"
end_date = "2045-12-31"
resolution = "1d"  # Change to "1h", "3h", "1w", etc., for different resolutions

# Build and visualize the S-curve
s_curve = build_s_curve(start_date, end_date, resolution)

# Plot the curve
plt.figure(figsize=(10, 6))
plt.plot(s_curve.index, s_curve.values, label="S-Curve", color="blue")
plt.axvline(pd.Timestamp("2024-01-01"), color="red", linestyle="--", label="2024")
plt.axvline(pd.Timestamp("2030-01-01"), color="green", linestyle="--", label="2030")
plt.scatter([pd.Timestamp("2024-01-01"), pd.Timestamp("2030-01-01")],
            [s_curve["2024-01-01"], s_curve["2030-01-01"]],
            color="black", label="2024/2030 Values")
plt.xlabel("Time")
plt.ylabel("S-Curve Value")
plt.title("S-Curve Visualization")
plt.legend()
plt.grid()
plt.show()


AttributeError: 'Index' object has no attribute 'mean'